In [29]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

In [30]:
df=pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,...,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,...,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,...,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,...,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,...,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,926424,M,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,...,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115,NaN
565,926682,M,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,...,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637,NaN
566,926954,M,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,...,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820,NaN
567,927241,M,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,...,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400,NaN


In [31]:
df.drop(columns=['id','Unnamed: 32'],inplace=True)

In [32]:
df.shape

(569, 31)

In [33]:
x_train,x_test,y_train,y_test=train_test_split(df.iloc[:,1:],df.iloc[:,0],test_size=0.2)

In [34]:
scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

In [35]:
labelencoder=LabelEncoder()
y_train=labelencoder.fit_transform(y_train)
y_test=labelencoder.transform(y_test)

In [36]:
x_train_tensor=torch.from_numpy(x_train)
x_test_tensor=torch.from_numpy(x_test)
y_train_tensor=torch.from_numpy(y_train)
y_test_tensor=torch.from_numpy(y_test)

In [37]:
x_train_tensor.shape
y_train_tensor.shape

torch.Size([455])

In [56]:
class SimpleNN():
  def __init__(self,x):
    self.weights=torch.rand(x.shape[1],1,dtype=torch.float64,requires_grad=True)
    self.bias=torch.zeros(1,dtype=torch.float64,requires_grad=True)

  def forward(self,x):
     z=torch.matmul(x,self.weights)+self.bias
     y_pred=torch.sigmoid(z)
     return y_pred

  def loss(self,y_pred,y):
    eplison=1e-15 #to prevent log(0)
    y_pred=torch.clamp(y_pred,eplison,1-eplison)
    loss=-(y*torch.log(y_pred)+(1-y)*torch.log(1-y_pred)).mean()
    return loss

In [50]:
learning_rate=0.1
epochs=25

In [64]:
#define model
model=SimpleNN(x_train_tensor)

#create loop
for epoch in range(epochs):
  #forward pass
  y_pred=model.forward(x_train_tensor)
  #loss calcuate
  Loss=model.loss(y_pred,y_train_tensor)
  #backpass
  Loss.backward()
  #parameters update
  with torch.no_grad():
   model.weights-=learning_rate*model.weights.grad
   model.bias-=learning_rate*model.bias.grad

  model.weights.grad.zero_()
  model.bias.grad.zero_()

  print(f'Epoch:{epoch+1},loss:{Loss.item()}')




Epoch:1,loss:4.42913931413846
Epoch:2,loss:4.23491829872788
Epoch:3,loss:4.041435694752717
Epoch:4,loss:3.841998559679028
Epoch:5,loss:3.6389432925306613
Epoch:6,loss:3.437855255528229
Epoch:7,loss:3.2372071019663626
Epoch:8,loss:3.03540541563525
Epoch:9,loss:2.836779285275075
Epoch:10,loss:2.6418199210991262
Epoch:11,loss:2.4511963887626576
Epoch:12,loss:2.26576006684127
Epoch:13,loss:2.0865633703201314
Epoch:14,loss:1.9148694734115723
Epoch:15,loss:1.7521417864075817
Epoch:16,loss:1.600000662276531
Epoch:17,loss:1.4601336886867808
Epoch:18,loss:1.3341551836794527
Epoch:19,loss:1.223428326110286
Epoch:20,loss:1.128858280427704
Epoch:21,loss:1.0506361695088893
Epoch:22,loss:0.9879801164209485
Epoch:23,loss:0.9390923233612204
Epoch:24,loss:0.9014989861457384
Epoch:25,loss:0.8725896652165377


In [65]:
model.weights

tensor([[-0.1865],
        [ 0.1305],
        [ 0.2279],
        [ 0.0707],
        [ 0.2231],
        [ 0.0033],
        [-0.2617],
        [-0.1847],
        [-0.2683],
        [ 0.4356],
        [ 0.3325],
        [ 0.4362],
        [ 0.0943],
        [-0.0063],
        [ 0.5630],
        [-0.5264],
        [ 0.2977],
        [ 0.2809],
        [ 0.2650],
        [-0.1904],
        [-0.4702],
        [-0.2205],
        [ 0.1856],
        [ 0.3685],
        [ 0.0081],
        [-0.1615],
        [ 0.1541],
        [ 0.2596],
        [-0.2586],
        [ 0.2684]], dtype=torch.float64, requires_grad=True)

In [66]:
model.bias

tensor([-0.1410], dtype=torch.float64, requires_grad=True)

In [71]:
with torch.no_grad():
  y_pred=model.forward(x_test_tensor)
  y_pred=(y_pred>0.7).float()
  accuray=(y_pred==y_test_tensor).float().mean()
  print(f'Accuracy:{accuray.item()}')

Accuracy:0.5812557935714722
